# 第 04 课 · 你的第一个真模拟（浏览器版）

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JavenZheng/monte-carlo-finance/blob/main/notebooks/04-your-first-real-simulation.ipynb)

配套课文：[docs/m1-tooling/04-your-first-real-simulation.md](../docs/m1-tooling/04-your-first-real-simulation.md)。

**怎么用**：在浏览器里打开 Colab 或 [JupyterLite](https://jupyter.org/try)，把这一页整份放进去，每一格按 `Shift + Enter` 运行。不需要安装任何东西，只用 Python 自带的 `random`。

**为什么每一段前面都有一行** `random.seed(7)`：钉住起点，同一份代码每次给你同一个数。课文里印的每个数字都是这样在本机抄下来的，你可以逐字核对。

## 一、把便利贴装进电脑

第 01 课你用六张纸演了十轮。这里把同一个世界写进程序：每条爆的概率六分之一，发六条，数一数有多少轮是**一条都没爆**。

先造两台机器（下面这一格只是定义，运行它不会打印任何东西）：

In [1]:
import random


def a_round(rate, posts):
    """发 posts 条，每条爆的概率是 rate。一条都没爆返回 True。"""
    hits = 0
    for _ in range(posts):
        if random.random() < rate:
            hits = hits + 1
    return hits == 0


def all_quiet_rate(rate, posts, runs):
    """重复 runs 轮，返回其中"全空"的轮占多大比例。"""
    quiet = 0
    for _ in range(runs):
        if a_round(rate, posts):
            quiet = quiet + 1
    return quiet / runs

## 二、同一道题，三档轮数

**注意每一行前面都重新钉了一次种子**：这样三行用的是同一条运气流的开头，你看到的差别只来自轮数，不来自运气。

课文里本机跑出来是 10 轮 20.0%、1000 轮 32.6%、100000 轮 33.4%（真值 33.49%）。

In [2]:
for runs in (10, 1000, 100000):
    random.seed(7)          # 每行都从同一条运气流开头
    p = all_quiet_rate(1 / 6, 6, runs)
    print(f"演 {runs} 轮 → 全空占 {p * 100:.1f}%")

演 10 轮 → 全空占 20.0%
演 1000 轮 → 全空占 32.6%
演 100000 轮 → 全空占 33.4%


## 三、造一个还没发生过的月

上一段算的是"概率是多少"。生活里你更常问的是另一类：**这个月我大概要花多少？最坏会到多少？**

这类问题没有公式，但你能一句一句造出来：每个工作日在 25 到 55 之间随挑一个数，挑 22 次，加总。一台新机器，塞进三个参数，吐出一个月的总额。

In [3]:
def one_month(low, high, workdays):
    """造一个没发生过的月：每个工作日的花销在 low 到 high 之间随机。"""
    total = 0
    for _ in range(workdays):
        total = total + random.uniform(low, high)
    return total

然后——这里是全部魔法所在——**把这台机器开一千次**，再把这一千个数从小到大排好。

本机跑出来：最省的月 737，最贵的月 1000，平均 879，一成的月低于 828，九成的月不超过 932。

In [4]:
random.seed(7)
months = []
for _ in range(1000):
    months.append(one_month(25, 55, 22))

months.sort()
p10 = months[int(0.1 * len(months))]
p90 = months[int(0.9 * len(months))]
print(f"最省的月 {months[0]:.0f}，最贵的月 {months[-1]:.0f}")
print(f"平均 {sum(months) / len(months):.0f}")
print(f"一成的月低于 {p10:.0f}，九成的月不超过 {p90:.0f}")

最省的月 737，最贵的月 1000
平均 879
一成的月低于 828，九成的月不超过 932


把这一千个月画出来，就是这个样子——横轴是这个月的总额，每一根蓝条是"有一千个月里的几个花成了这么多"，五条虚线就是上面那三行输出：

![一千个还没发生过的月](https://raw.githubusercontent.com/JavenZheng/monte-carlo-finance/main/assets/fig04_thousand_months.svg)

图由 [code/fig04_thousand_months.py](https://github.com/JavenZheng/monte-carlo-finance/blob/main/code/fig04_thousand_months.py) 生成，就是上面这一轮（种子 7、一千遍）。看两头那两根灰线：它们各自只压着一根几乎看不见的蓝条。


## 四、中间的数稳，两端的数飘

把下面这一格**连跑五遍**（这一格故意不钉种子）。你会看到最省和最贵那两个数每次都跳，而平均那一句几乎不动。

课文里我连跑五遍抄下的是：最省在 736 到 763 之间跳，最贵在 1000 到 1023 之间跳，平均五次是 879、880、880、881、881。**你跑出来的会是另一组数——这正是这一格要你看的东西。**

所以做预算时别拿"最贵那个月"当依据——那是千分之一的一条运气路径。要拿的是九成的那条线。

In [5]:
months = [one_month(25, 55, 22) for _ in range(1000)]
months.sort()
print(f"最省 {months[0]:.0f}，最贵 {months[-1]:.0f}，平均 {sum(months) / len(months):.0f}")

最省 759，最贵 1043，平均 880


## 五、今晚换成你自己的数

三个参数改完，你就有一千个还没发生的月：

- `low` / `high`：你每天最低、最高花多少
- `workdays`：一个月几个工作日

再往深里想一步：这一千个数里，**你报出口的是哪一个**？平均？九成的那条线？还是最贵那个？第 05 课会告诉你为什么"平均"这个选择本身就会骗人，第 07 课回答"这一千个数的平均可信到什么程度"。

---

**许可**：正文 CC BY-NC-SA 4.0，代码 MIT，见 [LICENSE.md](../LICENSE.md)。

**建议购买正版**：Paul Glasserman, *Monte Carlo Methods in Financial Engineering*, Springer, 2003；中译本《金融工程中的蒙特卡罗方法》。本 notebook 未复制原书任何文字、图表或表格。